<a href="https://colab.research.google.com/github/jnahMoch/revisedBookRecommender/blob/main/Books_categorizing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as npnp

booksdf = pd.read_csv("cleaned_books_dataset.csv")

In [2]:
booksdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4870 entries, 0 to 4869
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   isbn13               4870 non-null   int64  
 1   isbn10               4870 non-null   object 
 2   title                4870 non-null   object 
 3   authors              4870 non-null   object 
 4   categories           4870 non-null   object 
 5   description          4870 non-null   object 
 6   thumbnail            4870 non-null   object 
 7   average_rating       4870 non-null   float64
 8   num_pages            4870 non-null   float64
 9   cleaned_description  4867 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 380.6+ KB


In [3]:
booksdf.head(5)

,isbn13,isbn10,title,authors,categories,description,thumbnail,average_rating,num_pages,cleaned_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,A NOVEL THAT READERS and critics have been eag...,http://books.google.com/books/content?id=KQZCP...,3.85,247.0,novel reader critic eagerly anticipating decad...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Fiction,A new 'Christie for Christmas' -- a full-lengt...,http://books.google.com/books/content?id=gA5GP...,3.83,241.0,new christie christmas fulllength novel adapte...
2,9780006163831,0006163831,The One Tree,Stephen R. Donaldson,Fiction,Volume Two of Stephen Donaldson's acclaimed se...,http://books.google.com/books/content?id=OmQaw...,3.97,479.0,volume two stephen donaldsons acclaimed second...
3,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,"A memorable, mesmerizing heroine Jennifer -- b...",http://books.google.com/books/content?id=FKo2T...,3.93,512.0,memorable mesmerizing heroine jennifer brillia...
4,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Religion,Lewis' work on the nature of love divides love...,http://books.google.com/books/content?id=XhQ5X...,4.15,170.0,lewis work nature love divide love four catego...


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

booksdf['cleaned_description'] = booksdf['cleaned_description'].fillna('')

In [5]:
# Map existing metadata to your 4 target classes
category_mapping = {
    'Fiction': 'fiction',
    'Drama': 'fiction',
    'Comics & Graphic Novels': 'fiction',
    'Poetry': 'fiction',
    'Biography & Autobiography': 'Nonfiction',
    'History': 'Nonfiction',
    'Philosophy': 'Nonfiction',
    'Religion': 'Nonfiction',
    'Literary Criticism': 'Nonfiction',
    'Juvenile Fiction': "Children's fiction",
    'Juvenile Nonfiction': "Children's Nonfiction"
}
booksdf['simple_categories'] = booksdf['categories'].map(category_mapping)

In [6]:
# Split data into training and validation sets (80/20 split)
X = booksdf['cleaned_description']
y = booksdf['simple_categories']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [7]:
# Extract Text Features using TF-IDF Vectorizer
# We limit to the top 5,000 words and remove common English stop words
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [8]:
# Train a Logistic Regression Classifier
# 'class_weight="balanced"' helps handle categories with very few books (like Children's Nonfiction)
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [9]:
# Evaluate how well the model reads descriptions to predict categories
y_pred = model.predict(X_test_tfidf)
print("--- Model Performance Report ---")
print(classification_report(y_test, y_pred))

--- Model Performance Report ---
                       precision    recall  f1-score   support

Children's Nonfiction       0.41      0.32      0.36        22
   Children's fiction       0.54      0.65      0.59       105
           Nonfiction       0.70      0.80      0.75       233
              fiction       0.86      0.79      0.82       614

             accuracy                           0.77       974
            macro avg       0.63      0.64      0.63       974
         weighted avg       0.78      0.77      0.77       974



In [10]:
# Apply predictions across the entire dataset or save the file
booksdf.to_csv('books_with_simple_categories1.csv', index=False)

In [11]:
booksdf.to_pickle('books_with_simple_categories1.pkl')

print("Successfully converted CSV to Pickle format!")

Successfully converted CSV to Pickle format!
